### Data Quality Checks (Silver Layer)
Running this notebook to do some data quality/reconcilliation check ensuring no data is lost

#### Reconciliation

Ensuring nothing is lost between bronze and silver. 

- Bronze rows should equal valid_rows + quarantined_rows
- Deduped rows should equal distinct_valid_ids

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM dev_kyc.bronze.transactions)                 AS bronze_rows,
  (SELECT COUNT(*) FROM dev_kyc.silver.transactions_valid)           AS valid_rows,
  (SELECT COUNT(*) FROM dev_kyc.silver.transactions_quarantine)      AS quarantined_rows,
  (SELECT COUNT(DISTINCT txn_id) FROM dev_kyc.silver.transactions_valid) AS distinct_valid_ids,
  (SELECT COUNT(*) FROM dev_kyc.silver.transactions)      AS deduped_rows;

#### Rejected Rows


In [0]:
%sql
SELECT reject_reason, COUNT(*) AS rows
FROM dev_kyc.silver.transactions_quarantine
GROUP BY reject_reason
ORDER BY rows DESC;

#### Deduplication
The deduped table has no duplicate or null ids (both shoul return 0).

In [0]:
%sql
SELECT COUNT(*) - COUNT(DISTINCT txn_id) AS duplicate_ids,
       SUM(CASE WHEN txn_id IS NULL THEN 1 ELSE 0 END) AS null_ids
FROM dev_kyc.silver.transactions;

In [0]:
#### Type Check


In [0]:
%sql
DESCRIBE TABLE dev_kyc.silver.transactions;

#### Customers SCD1 vs SCD2 Check
 dev_kyc.silver.customers_current (SCD1) has one row per customer, dev_kyc.silver.customers_scd2 (SCD2) has history

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM dev_kyc.bronze.customers)                  AS bronze_versions,
  (SELECT COUNT(DISTINCT customer_id) FROM dev_kyc.bronze.customers) AS distinct_customers,
  (SELECT COUNT(*) FROM dev_kyc.silver.customers_current)          AS scd1_rows,
  (SELECT COUNT(*) FROM dev_kyc.silver.customers_scd2)             AS scd2_rows,
  (SELECT COUNT(*) FROM dev_kyc.silver.customers_scd2 WHERE __END_AT IS NULL) AS scd2_current_rows;

#### CUSTOMER SCD2 check
Side-by-side for customers that have more than one version.

In [0]:
%sql
SELECT customer_id, address, city, state, risk_rating, __START_AT, __END_AT
FROM dev_kyc.silver.customers_scd2
WHERE customer_id IN (
  SELECT customer_id FROM dev_kyc.silver.customers_scd2
  GROUP BY customer_id HAVING COUNT(*) > 1
)
ORDER BY customer_id, __START_AT
LIMIT 30;

In [0]:
%sql
SELECT * FROM dev_kyc.silver.customers_current WHERE customer_id = 'C00016';

In [0]:
%sql
SELECT * FROM dev_kyc.silver.customers_scd2    WHERE customer_id = 'C00016' ORDER BY __START_AT;
